In [2]:
# Loan Refinance Calculator
#
# NOTE: This functionality is also available in the combined program:
#   loan_refinance_toolkit.ipynb
# (tabs: "Single Refinance" and "Breakeven Rate Table")
#
# This program analyzes the refinancing of a loan (such as a mortgage).
#
# It is assumed that the period for the original loan (e.g., a month) is
# the same as the period for the new loan.

import tkinter as tk
import tkinter.ttk as ttk

root = tk.Tk()
root.title('Loan Refinancing Calculator')

# Bring the window to the front when it opens (works well from Jupyter).
root.lift()
try:
    root.attributes('-topmost', True)
    root.after(250, lambda: root.attributes('-topmost', False))
except tk.TclError:
    pass

def quit():
    root.destroy()
    
def _set_output(text: str):
    try:
        output_box.config(state='normal')
        output_box.delete('1.0', tk.END)
        output_box.insert(tk.END, text)
        output_box.config(state='disabled')
    except tk.TclError:
        pass


def clear():
    # This function clears the various input and output lines in the user interface
    q1.delete(0, tk.END)
    w.set(0)
    q2.delete(0, tk.END)
    x.set(0)
    q3.delete(0, tk.END)
    y.set(0)
    q4.delete(0, tk.END)
    z.set(0)

    q5.delete(0, tk.END)
    w1.set(0)
    q6.delete(0, tk.END)
    x1.set(0)
    q7.delete(0, tk.END)
    y1.set(0)

    _set_output('')


def get_input():
    # This function gets the input needed for the other functions
    global p0, i0, n0, n0_com, cost, i1, n1

    # Information for the initial loan that is to be refinanced
    p0 = w.get()       # principal for initial loan
    i0 = x.get()       # interest rate per period for initial loan
    n0 = y.get()       # number of periods for initial loan
    n0_com = z.get()   # number of periods completed when refinancing is to start

    # Information for refinancing
    cost = w1.get()    # cost of refinancing, incl termination + new-loan costs
    i1 = x1.get()      # interest rate per period for new loan
    n1 = y1.get()      # number of periods for new loan
    
    
def analyze():
# Function that analyzes a potential refinancing of a loan
    get_input()

    # Basic validation
    if p0 <= 0:
        _set_output('Error: please enter a positive initial loan amount.')
        return
    if n0 <= 0 or n1 <= 0:
        _set_output('Error: please enter positive values for the number of periods.')
        return
    if n0_com < 0 or n0_com > n0:
        _set_output('Error: completed periods must be between 0 and the original number of periods.')
        return
    if i0 < 0 or i1 < 0:
        _set_output('Error: interest rates must be nonnegative decimals (e.g., 0.005).')
        return

    # Compute equal payment amount (epa0) per period for initial loan
    if i0 == 0:
        epa0 = p0 / n0
        bal = max(0.0, p0 - epa0 * n0_com)
    else:
        g = (1 + i0) ** n0
        epa0 = p0 * i0 * g / (g - 1)
        # Balance of initial loan at the time of planned refinancing
        h = (1 + i0) ** n0_com
        bal = h * p0 - epa0 * (h - 1) / i0

    # Compute equal payment amount (epa1) per period for refinanced loan
    if bal <= 1e-6:
        _set_output('Loan is already paid off at the refinance point (balance is ~0).')
        return

    if i1 == 0:
        epa1 = bal / n1
    else:
        g1 = (1 + i1) ** n1
        epa1 = bal * i1 * g1 / (g1 - 1)

    # Comparison of remaining payments from the refinance decision point
    remaining_periods = n0 - n0_com
    t_pay0 = epa0 * remaining_periods
    interest0 = t_pay0 - bal

    # Refinance scenario
    t_pay1 = (epa1 * n1) + cost
    interest1 = (epa1 * n1) - bal

    # Time to recover refinancing cost
    if epa1 < epa0:
        monthly_savings = epa0 - epa1
        cost_recover = int(cost / monthly_savings) + 1 if cost > 0 else 0
        recover_line = f'It will take {cost_recover} periods to recover the cost of refinancing.'
    else:
        recover_line = (
            'The new periodic payment is not lower than the existing payment; '
            'refinancing may not make sense based on payment reduction alone.'
        )

    text = (
        f'The balance on the initial loan is ${bal:,.2f}\n\n'
        f'The periodic payment for the initial loan is ${epa0:,.2f}\n'
        f'The periodic payment for the new loan is ${epa1:,.2f}\n\n'
        f'The remaining payments for initial loan, if not refinanced, equal ${t_pay0:,.2f}\n'
        f'of which ${interest0:,.2f} are interest payments.\n\n'
        f'The payments in the refinancing scenario, which includes the costs to\n'
        f'terminate the initial loan and to initiate the new loan, equal ${t_pay1:,.2f}\n'
        f'The interest paid in this scenario (excluding refinancing costs) is ${interest1:,.2f}\n\n'
        f'{recover_line}'
    )
    _set_output(text)
        
#
# Input   
#
tk.Label(root, text="Amount of initial loan:").grid(row=1,column=1,padx=3,sticky='w')
w = tk.DoubleVar()
q1=tk.Entry(root, textvariable=w)
q1.grid(row=1, column=2,padx=5,pady=3,sticky='w')

tk.Label(root, text="Interest rate per period for initial loan:").grid(row=2,column=1,padx=3,sticky='w')
x = tk.DoubleVar()
q2=tk.Entry(root, textvariable=x)
q2.grid(row=2, column=2,padx=5,pady=3,sticky='w')

tk.Label(root, text="Number of periods for initial loan:").grid(row=3,column=1,padx=3,sticky='w')
y = tk.DoubleVar()
q3=tk.Entry(root, textvariable=y)
q3.grid(row=3, column=2,padx=5,pady=3,sticky='w')

tk.Label(root, text="Number of periods completed in initial \n loan when refinancing:").grid(row=4,column=1,padx=3,sticky='w')
z = tk.DoubleVar()
q4=tk.Entry(root, textvariable=z)
q4.grid(row=4, column=2,padx=5,pady=3,sticky='w')

# It is assumed that the entire remaining balance on the initial loan is to be refinanced

tk.Label(root, text="Penalities on termination of initial loan \n and financing costs for new loan:").grid(row=7,column=1,padx=3,sticky='w')
w1 = tk.DoubleVar()
q5=tk.Entry(root, textvariable=w1)
q5.grid(row=7, column=2,padx=5,pady=3,sticky='w')

tk.Label(root, text="Interest rate per period for refinanced loan:").grid(row=8,column=1,padx=3,sticky='w')
x1 = tk.DoubleVar()
q6=tk.Entry(root, textvariable=x1)
q6.grid(row=8, column=2,padx=5,pady=3,sticky='w')

tk.Label(root, text="Number of periods for refinanced loan:").grid(row=9,column=1,padx=3,sticky='w')
y1 = tk.DoubleVar()
q7=tk.Entry(root, textvariable=y1)
q7.grid(row=9, column=2,padx=5,pady=3,sticky='w')

# Control buttons

button1 = tk.Button(root, text="Analyze refinancing", command=analyze).grid(
    row=13, column=1, columnspan=2, padx=3, pady=7, sticky='nesw'
)

# Output area (shown in the GUI, not in the notebook cell output)
tk.Label(root, text='Output:').grid(row=15, column=1, padx=3, pady=(8, 2), sticky='w')
output_box = tk.Text(root, height=14, width=90)
output_box.grid(row=16, column=1, columnspan=2, padx=3, pady=(0, 8), sticky='w')
output_box.config(state='disabled')

button4 = tk.Button(root, text='Clear', command=clear).grid(row=23, column=1, pady=5)
button5 = tk.Button(root, text='Quit', command=quit).grid(row=23, column=2, padx=3, pady=5, sticky='w')

root.mainloop()